# BoW sweep analysis

Stateful notebook: one cell per sweep loads a variable; each plot is its own cell so you can tweak `y_exprs`, `show_seed_bar`, or `x_scale` independently. `plot_vs_epoch` draws per-epoch traces (for rollout sweeps, colored by rollouts, legend-grouped by corr, with only the highest-corr group visible by default — click any other group in the legend to reveal it). `plot_vs_corr` picks the argmax-over-epochs point per study and plots it against the dataset correlation.

In [1]:
from src import get_repo_base
from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

ARTIFACTS = get_repo_base() / "artifacts"
WRITEUP_ASSETS = get_repo_base() / "writeup" / "assets" / "bow"

# Per-epoch: two figures, each with two side-by-side panels.
Y_PER_EPOCH_RSQ = [
    rsq_expr(split="train", y="target"),          # left panel
    rsq_expr(split="train", y="ground_truth"),    # right panel
]
Y_PER_EPOCH_CORR = [
    corr_expr(split="train", y="ground_truth"),   # left panel
    corr_expr(split="val",   y="ground_truth"),   # right panel
]
# vs-corr: one subplot per expression (stacked vertically in uniform mode).
Y_VS_CORR = [
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val",   y="ground_truth"),
]


### Supervised-learning

In [2]:
sl = BagOfWordsAnalysisConfig.from_sl_sweep(
    study_base=ARTIFACTS / "bow-sl-sweep",
)
sl.describe("SL") if sl else print("SL: no artifacts")

display(sl.plot_vs_epoch(
    Y_PER_EPOCH_RSQ,
    title="SL: per-epoch (rsq)",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "sl_per_epoch_rsq.html",
))
display(sl.plot_vs_epoch(
    Y_PER_EPOCH_CORR,
    title="SL: per-epoch (corr)",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "sl_per_epoch_corr.html",
))
display(sl.plot_vs_corr(
    Y_VS_CORR,
    title="SL: best-epoch vs dataset_corr",
    x_scale="uniform",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "sl_vs_corr.html",
))


SL                                  32 runs    16 groups   up to 2 seeds/group


### GRPO artifacts

In [3]:
grpo = BagOfWordsAnalysisConfig.from_grpo_sweep(
    study_base=ARTIFACTS / "bow-grpo-sweep",
)
grpo.describe("GRPO") if grpo else print("GRPO: no artifacts")

display(grpo.plot_vs_epoch(
    Y_PER_EPOCH_RSQ,
    title="GRPO: per-epoch (rsq)",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "grpo_per_epoch_rsq.html",
))
display(grpo.plot_vs_epoch(
    Y_PER_EPOCH_CORR,
    title="GRPO: per-epoch (corr)",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "grpo_per_epoch_corr.html",
))
display(grpo.plot_vs_corr(
    Y_VS_CORR,
    title="GRPO: best-epoch vs dataset_corr",
    x_scale="uniform",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "grpo_vs_corr.html",
))


GRPO                                44 runs    44 groups   up to 1 seeds/group


### MaxRL artifacts

In [4]:
maxrl_sub = BagOfWordsAnalysisConfig.from_maxrl_sweep(
    study_base=ARTIFACTS / "bow-maxrl-sweep",
    subtract_baseline=True,
)
maxrl_sub.describe("MaxRL (subtract-baseline)") if maxrl_sub else print("MaxRL (subtract-baseline): no artifacts")

maxrl_nosub = BagOfWordsAnalysisConfig.from_maxrl_sweep(
    study_base=ARTIFACTS / "bow-maxrl-sweep",
    subtract_baseline=False,
)
maxrl_nosub.describe("MaxRL (no-subtract-baseline)") if maxrl_nosub else print("MaxRL (no-subtract-baseline): no artifacts")


MaxRL (subtract-baseline)            7 runs     7 groups   up to 1 seeds/group
MaxRL (no-subtract-baseline): no artifacts


In [5]:
display(maxrl_sub.plot_vs_epoch(
    Y_PER_EPOCH_RSQ,
    title="MaxRL: Rsq by epoch",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "maxrl_sub_per_epoch_rsq.html",
))
display(maxrl_sub.plot_vs_epoch(
    Y_PER_EPOCH_CORR,
    title="MaxRL: Corr by epoch",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "maxrl_sub_per_epoch_corr.html",
))
display(maxrl_sub.plot_vs_corr(
    Y_VS_CORR,
    title="MaxRL (subtract-baseline): best-epoch vs dataset_corr",
    x_scale="uniform",
    show_seed_bar=True,
    save_path=WRITEUP_ASSETS / "maxrl_sub_vs_corr.html",
))


In [6]:
# display(
#     maxrl_nosub.plot_vs_epoch(
#         Y_EXPRS,
#         title="MaxRL (no-subtract-baseline): per-epoch",
#         show_seed_bar=True,
#     )
# )
# display(
#     maxrl_nosub.plot_vs_corr(
#         Y_EXPRS,
#         title="MaxRL (no-subtract-baseline): best-epoch vs dataset_corr",
#         x_scale="uniform",
#         show_seed_bar=True,
#     )
# )